# Subqueries and EXISTS

Practice notebook for the datafromzero SQL lesson **Subqueries and EXISTS**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. IN: the inner query alone, then inside the outer query

In [ ]:
con.sql("""
SELECT customer_id
FROM orders
WHERE store_id = 3
ORDER BY customer_id
LIMIT 8
""").show()

In [ ]:
con.sql("""
SELECT customer_id, signup_date, loyalty_tier
FROM customers
WHERE customer_id IN (
  SELECT customer_id
  FROM orders
  WHERE store_id = 3
)
ORDER BY customer_id
LIMIT 6
""").show()

### Step 2. NOT EXISTS: customers with no order

In [ ]:
con.sql("""
SELECT customer_id, signup_date, loyalty_tier
FROM customers c
WHERE NOT EXISTS (
  SELECT 1
  FROM orders o
  WHERE o.customer_id = c.customer_id
)
ORDER BY customer_id
""").show()

### Step 3. Read a correlated subquery in the `SELECT` list

In [ ]:
con.sql("""
SELECT c.customer_id,
       (SELECT COUNT(*) FROM orders o WHERE o.customer_id = c.customer_id) AS orders
FROM customers c
WHERE c.customer_id BETWEEN 138 AND 142
ORDER BY c.customer_id
""").show()

### Step 4. A derived table: the average of a count

In [ ]:
con.sql("""
SELECT ROUND(AVG(orders), 2) AS avg_orders, MAX(orders) AS max_orders
FROM (
  SELECT customer_id, COUNT(*) AS orders
  FROM orders
  WHERE customer_id IS NOT NULL
  GROUP BY customer_id
) AS per_customer
""").show()

### Step 5. Scalar: customers above the average

In [ ]:
con.sql("""
SELECT ROUND(COUNT(customer_id) * 1.0 / COUNT(DISTINCT customer_id), 2) AS avg_orders
FROM orders
""").show()

In [ ]:
con.sql("""
SELECT customer_id, COUNT(*) AS orders
FROM orders
WHERE customer_id IS NOT NULL
GROUP BY customer_id
HAVING COUNT(*) > (
  SELECT ROUND(COUNT(customer_id) * 1.0 / COUNT(DISTINCT customer_id), 2)
  FROM orders
)
ORDER BY orders DESC, customer_id
LIMIT 5
""").show()

### Step 6. Verify

In [ ]:
con.sql("""
SELECT
  (SELECT COUNT(*) FROM customers
   WHERE customer_id IN (SELECT customer_id FROM orders WHERE store_id = 3)) AS at_harbor,
  (SELECT COUNT(*) FROM customers c
   JOIN orders o ON o.customer_id = c.customer_id
   WHERE o.store_id = 3) AS join_rows,
  (SELECT COUNT(*) FROM customers c
   WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)) AS ordered,
  (SELECT COUNT(*) FROM customers c
   WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)) AS never
""").show()

## Exercise 1. Average and most lines on an order

The outer query is done for you. Finish the derived table: one row per order, with its number of lines in a column named
`lines`. Expected: 1 row, 2 columns.

In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    con.sql("""
    -- replace the ... with the derived table, then run it
    SELECT ROUND(AVG(lines), 2) AS avg_lines, MAX(lines) AS max_lines
    FROM (
      ...
    ) AS per_order
    """).show()
except Exception as e:
    print(e)


## Exercise 2. Customers who placed an order but never a completed one

Before you run anything, write one sentence in the cell below: which customers you expect, and how they differ from the
five who never ordered. Then write the query with `EXISTS` and `NOT EXISTS`, and try the `NOT EXISTS` half alone.

In [ ]:
# My expected result, in words:
#

In [ ]:
# Query: an order exists, and no completed order exists

In [ ]:
# The NOT EXISTS half alone, to compare

## Exercise 3. Spot the flaw

A colleague wants the customers who never ordered. Run the query, then find out what went wrong and write a query that
gives the right customers.

In [ ]:
con.sql("""
SELECT customer_id
FROM customers
WHERE customer_id NOT IN (SELECT customer_id FROM orders)
""").show()

In [ ]:
# Your corrected query: